google collab depedencies

In [1]:
!pip -q install bertopic
!pip -q install sastrawi
!pip -q install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 47.1 MB/s eta 0:00:00


In [2]:
!git clone -q -b gavriel-thesis https://github.com/ranslemus/topic_modeling_KBMI4.git
%cd topic_modeling_KBMI4

/content/topic_modeling_KBMI4


In [3]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px
import random

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from hdbscan.validity import validity_index

# for linux
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

# for windows
# import umap as UMAP
# import hdbscan as HDBSCAN

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : Tesla T4


In [5]:
df = pd.read_csv("data/preprocessed_data_downsampled.csv")

df.head()

,reviewId,bank,score,year,text
0,96c7507f-8be5-40a9-8b3a-009ac2036630,WONDR_BNI_REVIEWS,2,2024,kebanyakan maintenance jadi enggak efektif pak...
1,e0c5d420-5bb9-4408-854e-4f6744dd432b,WONDR_BNI_REVIEWS,1,2024,sampai saat ini tidak bisa diakses sudah dinon...
2,f977cd45-7e77-4735-a7de-7e4e8b827db3,LIVIN_MANDIRI_REVIEWS,1,2023,enggak bisa dibuka padahal usah coba berbagai ...
3,629f06db-dc19-4a6b-a526-c5fa09933ed2,LIVIN_MANDIRI_REVIEWS,2,2025,kenapa di login tidak bisa ya malah muncul tul...
4,d1116af0-d952-49b8-9ff9-692f6093f599,BRIMO_REVIEWS,1,2023,buruk data sudah benar malah enggak bisa dikon...


In [6]:
df["word_count"] = df["text"].astype(str).str.split().apply(len)
df = df[df["word_count"] >= 5].reset_index(drop=True)
print(f"Total documents setelah filter: {len(df):,}")

Total documents setelah filter: 135,913


In [7]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 135,913


# IndoBERT

In [9]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)

model.eval()

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(50000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [10]:
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

In [11]:
def encode_documents(
    documents,
    batch_size=32,
    max_length=128
):

    embeddings = []

    with torch.no_grad():

        for i in tqdm(
            range(0, len(documents), batch_size)
        ):

            batch = documents[
                i:i+batch_size
            ]

            encoded_input = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            encoded_input = {
                k: v.to(device)
                for k, v in encoded_input.items()
            }

            model_output = model(**encoded_input)

            sentence_embeddings = mean_pooling(
                model_output,
                encoded_input["attention_mask"]
            )

            sentence_embeddings = (
                sentence_embeddings
                .cpu()
                .numpy()
            )

            embeddings.append(sentence_embeddings)

    return np.vstack(embeddings)

In [12]:
embeddings = encode_documents(
    documents,
    batch_size=32,
    max_length=128
)

  0%|          | 0/4248 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

In [13]:
print(embeddings.shape)

(135913, 768)


In [14]:
embeddings[0]

array([ 1.99362922e+00,  8.71141791e-01,  2.32576132e-01,  4.88159537e-01,
       -1.53942794e-01,  6.54067397e-02, -7.94849396e-01,  6.44384176e-02,
        5.35462677e-01, -1.38591528e-01, -8.13294351e-02, -1.04618895e+00,
       -9.04206634e-01,  8.62347782e-01,  2.20925231e-02, -3.00665766e-01,
       -5.28008997e-01, -1.01320833e-01, -2.91478708e-02,  7.27127671e-01,
        3.72639418e-01, -3.66047174e-01, -6.55413792e-02, -1.06161618e+00,
       -7.23849773e-01, -2.64187664e-01, -7.38127232e-02,  3.38941991e-01,
       -4.32386130e-01, -4.99101728e-01,  7.13105083e-01,  4.50596094e-01,
       -7.68143870e-03,  3.48131716e-01, -1.58587146e+00,  1.17839050e+00,
       -4.46127206e-01,  1.15841889e+00, -1.05954075e+00, -1.42351031e-01,
       -1.08025956e+00,  3.64045143e-01, -1.19255567e+00, -4.37219381e-01,
       -3.45705301e-01,  7.09880710e-01,  3.18425953e-01,  1.71062803e+00,
        4.44132164e-02,  1.10976048e-01, -1.06760168e+00, -7.84462571e-01,
        2.28673473e-01,  

In [15]:
norms = np.linalg.norm(embeddings, axis=1)

print("Minimum Norm :", norms.min())
print("Maximum Norm :", norms.max())
print("Average Norm :", norms.mean())
print("Std Norm :", norms.std())

Minimum Norm : 12.87398
Maximum Norm : 26.059353
Average Norm : 17.716616
Std Norm : 1.4305303


In [16]:
print("NaN :", np.isnan(embeddings).sum())
print("Inf :", np.isinf(embeddings).sum())

NaN : 0
Inf : 0


In [25]:
np.save(
    "indobert_embeddings_downsampled.npy",
    embeddings
)

# BERTopic

In [127]:
embeddings = np.load("indobert_embeddings_downsampled.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (135913, 768)


In [128]:
sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

# extra_particles = ["banget", "terus", "padahal", "sih", "aja", "saja", "dong", "deh", "ya", "kok", "biar", "gitu", "nih", "loh", "mau", "sudah", "belum"]
# sastrawi_stopwords_extended = sastrawi_stopwords + extra_particles

vectorizer_model = CountVectorizer(
  ngram_range=(1,2),
  stop_words=sastrawi_stopwords,
  token_pattern=r"(?u)\b[^\d\W]+\b",
  min_df=2,
  )

baseline UMAP for testing purpose

In [129]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [130]:
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

In [131]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
)

In [132]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-08-10 08:51:53,685 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-10 08:52:04,932 - BERTopic - Dimensionality - Completed ✓
2026-08-10 08:52:04,935 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-10 08:52:07,834 - BERTopic - Cluster - Completed ✓
2026-08-10 08:52:07,860 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-10 08:52:11,853 - BERTopic - Representation - Completed ✓


# Evaluation

Basic Statistics

In [133]:
topic_info = topic_model.get_topic_info()

topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,0,135783,0_enggak_aplikasi_nya_mau,"[enggak, aplikasi, nya, mau, terus, padahal, u...",[perbaiki lagi apk nya mau transfer susah kada...
1,1,68,1_unable_identity_your_your identity,"[unable, identity, your, your identity, authen...",[kenapa bca mobile unable tapi authenticate yo...
2,2,62,2_the_and_is_i,"[the, and, is, i, this, not, a, my, applicatio...",[the developer and tester need tapi be replace...


In [134]:
num_topics = len(topic_info) - 1

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 2
Outliers            : 0
Outlier Percentage  : 0.00%


Topic Size

In [135]:
topic_info[["Topic","Count"]]

,Topic,Count
0,0,135783
1,1,68
2,2,62


Top Words

In [136]:
for topic in topic_info.Topic:

    if topic == -1:
        continue

    print("="*80)

    print(f"Topic {topic}")

    print(topic_model.get_topic(topic))

Topic 0
[('enggak', np.float64(0.11481987346801341)), ('aplikasi', np.float64(0.07059790769656339)), ('nya', np.float64(0.06662076511752367)), ('mau', np.float64(0.06342990617499039)), ('terus', np.float64(0.05762258863572465)), ('padahal', np.float64(0.05248659982382046)), ('update', np.float64(0.051247806611402436)), ('malah', np.float64(0.04615429578115881)), ('masuk', np.float64(0.04451402615764246)), ('gagal', np.float64(0.04373317787393787))]
Topic 1
[('unable', np.float64(0.36723906050039923)), ('identity', np.float64(0.3617862593056985)), ('your', np.float64(0.358359152032734)), ('your identity', np.float64(0.34919954760458377)), ('authenticate', np.float64(0.3272444558554711)), ('authenticate your', np.float64(0.31119369878169073)), ('unable authenticate', np.float64(0.305633878631753)), ('tulisan unable', np.float64(0.11716188574520404)), ('tulisan', np.float64(0.0748456907380544)), ('muncul', np.float64(0.04665496553373137))]
Topic 2
[('the', np.float64(0.3290573400828392)),

Representative Reviews

In [137]:
representative_docs = topic_model.get_representative_docs()

for topic in representative_docs:

    if topic == -1:
        continue

    print("="*100)

    print(f"Topic {topic}")

    print()

    for doc in representative_docs[topic][:5]:

        print("-", doc)

    print()

Topic 0

- perbaiki lagi apk nya mau transfer susah kadang ganguan saldo rek enggak ada aneh livin kalau mau memudahkan transaksi enggak usah ada livin lah
- aplikasi nya makin lama bikin emosi di hp lama padahal livin nya sudah terdaftar giliran saya beli hp baru ku pindahkan gagal terus katanya tanggal lahir salah padahal sebelumnya sudah bisa login livin di hp lama benar benar ajaib enggak jelas kalau enggak niat bikin sama banking enggak usahlah bikin jadi kecewa kan nasabah nya kalau begini
- mau transaksi enggak bisa padahal pin sudah betul nanti sudah ganti pin password lagi yang enggak bisa loging padahal password sudah betul nanti sudah ganti password pin transaksi lagi yang enggak bisa berulang ulang begitu ke bank cs juga enggak bisa apa-apa enggak ada solusi aplikasi enggak bagus sama banking lama sangat memuaskan beda sama aplikasi bank sebelah makanya jarang sudah pakai bni aplikasi sebelah sangat bagus enggak lama banyak orang pindah ke bank sebelah

Topic 1

- kenapa bc

silhoutte score

In [ ]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask]
)

print(f"Silhouette Score : {silhouette:.4f}")

In [ ]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info.Topic:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]
    topic_words.append(words)

unique_words = len(
    set(chain.from_iterable(topic_words))
)

total_words = len(topic_words) * top_n
topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Representative Reviews

In [ ]:
representative_docs = topic_model.get_representative_docs()

for topic, docs in representative_docs.items():

    if topic == -1:
        continue

    print("="*100)

    print(f"TOPIC {topic}")

    print()

    for i, doc in enumerate(docs[:5],1):

        print(f"{i}. {doc}")

        print()

In [ ]:
random.seed(42)

sample_size = 20

for topic_id in sorted(set(topics)):

    if topic_id == -1:
        continue

    topic_docs = [
        doc for doc, topic in zip(documents, topics)
        if topic == topic_id
    ]

    n = min(sample_size, len(topic_docs))
    sampled_docs = random.sample(topic_docs, n)

    print("\n" + "=" * 120)
    print(f"TOPIC {topic_id}")
    print(f"CLUSTER SIZE : {len(topic_docs)}")
    print(f"SAMPLE SIZE  : {n}")
    print("=" * 120)

    for i, doc in enumerate(sampled_docs, 1):
        print(f"{i}. {doc}")

NPMI

In [ ]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [ ]:
doc.split()

In [ ]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [ ]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)
topic_words = []

for topic in topic_info.Topic:

    if topic == -1:
        continue

    words = []

    for word, score in topic_model.get_topic(topic):
        if word in dictionary.token2id:
            words.append(word)
    # Need at least 2 words for coherence
    if len(words) >= 2:
        topic_words.append(words)

In [ ]:
# sanity check
print(f"Valid Topics : {len(topic_words)}")

print()

print(topic_words[:3])

In [ ]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

DBCV

In [ ]:
mask = np.array(topics) != -1
X = topic_model.umap_model.embedding_[mask].astype(np.float64)
labels = np.array(topics)[mask]

dbcv_score = validity_index(X, labels)
print(f"DBCV : {dbcv_score:.4f}")

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

df["topic"] = topics

# 1. Baseline: proporsi tiap bank di keseluruhan korpus
baseline = df["bank"].value_counts(normalize=True) * 100
print("Proporsi bank di keseluruhan korpus (baseline):")
print(baseline.round(2))
print()

# 2. Proporsi tiap bank DI DALAM tiap topik
crosstab = pd.crosstab(df["topic"], df["bank"], normalize="index") * 100
crosstab = crosstab.round(2)

# 3. Hitung "lift" = proporsi di topik / proporsi baseline
#    >1 artinya over-represented di topik itu, <1 artinya under-represented
lift = crosstab.copy()
for bank in baseline.index:
    lift[bank] = crosstab[bank] / baseline[bank]

# 4. Tandai topik yang "njomplang" (deviasi lift > 1.5x atau < 0.5x dari baseline)
def flag_imbalance(row):
    return any(row > 1.5) or any(row < 0.5)

lift["is_imbalanced"] = lift[baseline.index].apply(flag_imbalance, axis=1)

# gabung count per topik biar gampang liat mana yang topik "besar" (bukan cuma noise kecil)
topic_sizes = df[df["topic"] != -1]["topic"].value_counts()
lift["topic_size"] = lift.index.map(topic_sizes)

result = lift[lift.index != -1].sort_values("is_imbalanced", ascending=False)
print(result[list(baseline.index) + ["is_imbalanced", "topic_size"]])